# Episode 2 — JIT, Tracing, and the Jaxpr

**Instructor notebook** · run top-to-bottom before recording.

`jax.jit` traces your function once into a **jaxpr**, lowers to XLA, and caches the executable.

| | |
|---|---|
| **Chapter** | 1.2 · Part I — Pure JAX |
| **Prereq** | Episode 1 |
| **Next** | Episode 3 — automatic differentiation |

**JAX docs:** [Tracing](https://docs.jax.dev/en/latest/tracing.html) · [jaxpr](https://docs.jax.dev/en/latest/jaxpr.html) · [`jax.make_jaxpr`](https://docs.jax.dev/en/latest/_autosummary/jax.make_jaxpr.html) · [`jax.jit`](https://docs.jax.dev/en/latest/_autosummary/jax.jit.html)


In [ ]:
import time

import jax
import jax.numpy as jnp
import jax.random as jr
from jax import make_jaxpr

## What `jit` does

On the first call with given **shapes/dtypes**, JAX **traces** your function: values become tracers, ops record into a **jaxpr**. That jaxpr lowers to **StableHLO**, XLA optimizes it, and later calls reuse the cached executable. New shapes → **retrace**.

## A 3-layer MLP to inspect

In [ ]:
def mlp3(params, x):
    for layer in params:
        x = jnp.tanh(x @ layer["w"] + layer["b"])
    return x


def init_mlp(key, d_in, d_hidden, d_out):
    key, k0, k1, k2 = jr.split(key, 4)
    return [
        {"w": jr.normal(k0, (d_in, d_hidden)) * 0.1, "b": jnp.zeros(d_hidden)},
        {"w": jr.normal(k1, (d_hidden, d_hidden)) * 0.1, "b": jnp.zeros(d_hidden)},
        {"w": jr.normal(k2, (d_hidden, d_out)) * 0.1, "b": jnp.zeros(d_out)},
    ]


key = jr.key(0)
params = init_mlp(key, 16, 32, 4)
x = jr.normal(jr.key(1), (8, 16))

print(make_jaxpr(mlp3)(params, x))

## `jit` wraps the jaxpr in a compiled call

In [ ]:
jitted_mlp = jax.jit(mlp3)
print(make_jaxpr(jitted_mlp)(params, x))

## First call vs steady state

In [ ]:
def time_call(fn, *args):
    t0 = time.perf_counter()
    out = jax.block_until_ready(fn(*args))
    return (time.perf_counter() - t0) * 1000, out


fresh = jax.jit(mlp3)
first_ms, _ = time_call(fresh, params, x)
for _ in range(3):
    fresh(params, x)
steady_ms, _ = time_call(fresh, params, x)
print(f"first call (compile+run): {first_ms:.2f} ms")
print(f"steady state (run only):  {steady_ms:.2f} ms")

## Retrace on shape change

In [ ]:
compile_log = []


def mlp_trace(params, x):
    compile_log.append(x.shape)
    return mlp3(params, x)


jitted_trace = jax.jit(mlp_trace)
x_a = jr.normal(jr.key(2), (4, 16))
x_b = jr.normal(jr.key(3), (8, 16))

jitted_trace(params, x_a)
jitted_trace(params, x_a)
jitted_trace(params, x_b)
print("shapes seen at trace time:", compile_log)

## Static vs traced — what `jit` can and cannot specialize on

| Kind | Examples | At `jit` time |
|------|----------|----------------|
| **Traced** | `jnp` arrays | Shapes and dtypes are baked into the compiled program. New shapes → **retrace**. |
| **Static** | Python `int`, `bool`, `str`, `None` that control structure | Must mark with `static_argnums` / `static_argnames` or they become tracers and break Python control flow. |
| **Not compiled** | `print`, file I/O, mutation, arbitrary Python loops over Python lists | Run at **trace** time only (or not at all inside compiled code). |

**Rule of thumb:** tensor data → traced; hyperparameters and flags that pick branches → static.

## `static_argnums` for Python scalars

In [ ]:
def repeat_matmul(x, n):
    for _ in range(n):
        x = x @ x
    return x


# n is a Python int — mark static so changing n does not confuse the batch dim
jitted_repeat = jax.jit(repeat_matmul, static_argnums=1)

a = jr.normal(jr.key(4), (64, 64))
print("n=2:", jitted_repeat(a, 2).shape)
print("n=3:", jitted_repeat(a, 3).shape)

## Print fires at **trace** time, not every run

In [ ]:
print("--- outside jit ---")


@jax.jit
def noisy_square(x):
    print("inside jitted function, x.shape =", x.shape)
    return x ** 2


v = jnp.ones((3,))
print("run 1:")
noisy_square(v)
print("run 2 (same shape):")
noisy_square(v)
print("run 3 (new shape):")
noisy_square(jnp.ones((5,)))

> **Key insight:** JIT does not execute your Python code at runtime — it traces it once. Print statements inside jitted functions fire at trace time, not run time.

---
## Exercise

1. `make_jaxpr` on the 3-layer MLP — find a `dot_general` or `dot` primitive.
2. Change batch size and log how many distinct shapes your jitted function sees.
3. Pass a Python `int` loop count with `static_argnums`.
4. Time first vs second `jit` call on the same shapes.
5. List one traced argument and one static argument from your MLP example.

*(Solution below.)*

In [ ]:
print("jaxpr snippet:")
print(str(make_jaxpr(jitted_mlp)(params, x))[:400], "...")

**Next:** Episode 3 — `grad`, `value_and_grad`, and checkpointing.